# 03b - Feature Selection and Hyperparameter Optimization

This notebook performs joint optimization of:

- LSTM hyperparameters
- feature selection
- architecture configuration

The optimization is performed using manually implemented metaheuristics without external optimization libraries.

In [1]:
import os
import json
import time
import random
import gc
import joblib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam, RMSprop

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

I0000 00:00:1779417545.592665  628404 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779417545.617675  628404 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779417546.159808  628404 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
gpus = tf.config.list_physical_devices("GPU")

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)

        print("Memory growth enabled.")

    except RuntimeError as e:
        print(e)

print("Physical GPUs:", tf.config.list_physical_devices("GPU"))
print("Logical GPUs:", tf.config.list_logical_devices("GPU"))

Physical GPUs: []
Logical GPUs: []


E0000 00:00:1779417546.559452  628404 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
I0000 00:00:1779417546.559469  628404 cuda_diagnostics.cc:171] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
I0000 00:00:1779417546.559470  628404 cuda_diagnostics.cc:176] retrieving CUDA diagnostic information for host: e2fef2328636
I0000 00:00:1779417546.559472  628404 cuda_diagnostics.cc:183] hostname: e2fef2328636
I0000 00:00:1779417546.559505  628404 cuda_diagnostics.cc:190] libcuda reported version is: 595.58.3
I0000 00:00:1779417546.559510  628404 cuda_diagnostics.cc:194] kernel reported version is: 595.58.3
I0000 00:00:1779417546.559511  628404 cuda_diagnostics.cc:284] kernel version seems to match DSO: 595.58.3


In [3]:
def set_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

In [4]:
BASE_DIR = "/home/jovyan/work"

OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
ARRAYS_DIR = os.path.join(OUTPUT_DIR, "arrays")
SCALERS_DIR = os.path.join(OUTPUT_DIR, "scalers")
RESULTS_DIR = os.path.join(OUTPUT_DIR, "results")
MODELS_DIR = os.path.join(OUTPUT_DIR, "feature_selected_models")
FIGURES_DIR = os.path.join(OUTPUT_DIR, "figures")

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

In [5]:
config_path = os.path.join(OUTPUT_DIR, "preprocessing_config.json")

with open(config_path, "r") as f:
    CONFIG = json.load(f)

CONFIG

{'file_name': 'EMA--OMUAQ_Table1.txt',
 'timestamp_column': 'TIMESTAMP',
 'target': 'SolRad_Avg',
 'window_size': 12,
 'horizon': 6,
 'train_ratio': 0.7,
 'val_ratio': 0.15,
 'test_ratio': 0.15,
 'irradiance_threshold': 50,
 'features': ['TempAir_Avg',
  'HumRel',
  'PreBar_Avg',
  'VelVto_S_WVT',
  'DirVto_D1_WVT',
  'DirVto_SD1_WVT',
  'hour',
  'dayofyear',
  'month'],
 'n_features': 9,
 'n_samples': 28034,
 'train_size': 19623,
 'val_size': 4205,
 'test_size': 4206}

In [6]:
X_train = np.load(os.path.join(ARRAYS_DIR, "X_train.npy"))
y_train = np.load(os.path.join(ARRAYS_DIR, "y_train.npy"))

X_val = np.load(os.path.join(ARRAYS_DIR, "X_val.npy"))
y_val = np.load(os.path.join(ARRAYS_DIR, "y_val.npy"))

X_test = np.load(os.path.join(ARRAYS_DIR, "X_test.npy"))
y_test = np.load(os.path.join(ARRAYS_DIR, "y_test.npy"))

daylight_test = np.load(os.path.join(ARRAYS_DIR, "daylight_test.npy"))

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (19623, 12, 9)
X_val: (4205, 12, 9)
X_test: (4206, 12, 9)


In [7]:
scaler_y = joblib.load(
    os.path.join(SCALERS_DIR, "scaler_y.pkl")
)

In [8]:
FEATURE_NAMES = CONFIG["features"]

for i, feature in enumerate(FEATURE_NAMES):
    print(i, feature)

0 TempAir_Avg
1 HumRel
2 PreBar_Avg
3 VelVto_S_WVT
4 DirVto_D1_WVT
5 DirVto_SD1_WVT
6 hour
7 dayofyear
8 month


In [9]:
WINDOW_SIZE = X_train.shape[1]
N_FEATURES = X_train.shape[2]

OPTIMIZATION_EPOCHS = 8
FINAL_EPOCHS = 80

PATIENCE_OPT = 3
PATIENCE_FINAL = 8

SEED = 42
set_seed(SEED)

print("Window size:", WINDOW_SIZE)
print("Number of features:", N_FEATURES)

Window size: 12
Number of features: 9


In [10]:
SEARCH_SPACE = {
    "units_1": [32, 64, 96],
    "units_2": [16, 32, 64],
    "dropout_rate": [0.1, 0.2, 0.3],
    "learning_rate": [0.0001, 0.0005, 0.001],
    "batch_size": [16, 32],
    "optimizer": ["adam"],
    "activation": ["relu"],
    "n_layers": [1, 2],
}

In [11]:
PARAMETER_NAMES = list(SEARCH_SPACE.keys())

N_HYPERPARAMS = len(PARAMETER_NAMES)
N_FEATURE_MASK = N_FEATURES
SOLUTION_SIZE = N_HYPERPARAMS + N_FEATURE_MASK

LOWER_BOUNDS = np.zeros(SOLUTION_SIZE)

UPPER_BOUNDS = np.array(
    [len(SEARCH_SPACE[param]) - 1 for param in PARAMETER_NAMES]
    + [1 for _ in range(N_FEATURE_MASK)]
)

print("Solution size:", SOLUTION_SIZE)
print("Hyperparameters:", N_HYPERPARAMS)
print("Feature mask size:", N_FEATURE_MASK)

Solution size: 17
Hyperparameters: 8
Feature mask size: 9


In [12]:
def get_optimizer(optimizer_name, learning_rate):
    if optimizer_name == "adam":
        return Adam(learning_rate=learning_rate)

    if optimizer_name == "rmsprop":
        return RMSprop(learning_rate=learning_rate)

    raise ValueError(f"Unsupported optimizer: {optimizer_name}")

In [13]:
def build_lstm_model(params, input_shape):
    model = Sequential()

    n_layers = params["n_layers"]

    if n_layers == 1:
        model.add(
            LSTM(
                params["units_1"],
                input_shape=input_shape
            )
        )

    else:
        model.add(
            LSTM(
                params["units_1"],
                return_sequences=True,
                input_shape=input_shape
            )
        )

        model.add(Dropout(params["dropout_rate"]))

        for layer_idx in range(2, n_layers):
            model.add(
                LSTM(
                    params["units_2"],
                    return_sequences=True
                )
            )

            model.add(Dropout(params["dropout_rate"]))

        model.add(
            LSTM(
                params["units_2"]
            )
        )

    model.add(Dropout(params["dropout_rate"]))

    model.add(
        Dense(
            32,
            activation=params["activation"]
        )
    )

    model.add(Dense(1))

    model.compile(
        optimizer=get_optimizer(
            params["optimizer"],
            params["learning_rate"]
        ),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [14]:
def inverse_transform(data):
    return scaler_y.inverse_transform(data)


def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }


def calculate_daylight_metrics(y_true, y_pred, daylight_mask):
    y_true_day = y_true[daylight_mask]
    y_pred_day = y_pred[daylight_mask]

    return calculate_metrics(y_true_day, y_pred_day)

In [ ]:
def apply_feature_mask(X, feature_mask):
    feature_mask = np.array(feature_mask).astype(bool)
    return X[:, :, feature_mask]
def decode_solution(solution):
    solution = np.clip(solution, LOWER_BOUNDS, UPPER_BOUNDS)

    params = {}

    for i, param_name in enumerate(PARAMETER_NAMES):
        option_index = int(round(solution[i]))

        option_index = np.clip(
            option_index,
            0,
            len(SEARCH_SPACE[param_name]) - 1
        )

        params[param_name] = SEARCH_SPACE[param_name][option_index]

    raw_mask = solution[N_HYPERPARAMS:]
    feature_mask = raw_mask >= 0.5

    if feature_mask.sum() == 0:
        feature_mask[np.argmax(raw_mask)] = True

    selected_features = [
        feature for feature, selected in zip(FEATURE_NAMES, feature_mask)
        if selected
    ]

    params["feature_mask"] = feature_mask
    params["selected_features"] = selected_features
    params["n_selected_features"] = int(feature_mask.sum())

    return params

def tournament_selection(population, fitness, k=3):
    indices = np.random.choice(len(population), size=k, replace=False)
    best_idx = indices[np.argmin(fitness[indices])]
    return population[best_idx].copy()

def crossover(parent1, parent2, alpha=0.5):
    d = np.abs(parent1 - parent2)
    lo = np.minimum(parent1, parent2) - alpha * d
    hi = np.maximum(parent1, parent2) + alpha * d
    child1 = lo + np.random.random(len(parent1)) * (hi - lo)
    child2 = lo + np.random.random(len(parent2)) * (hi - lo)
    return child1, child2

def mutate(individual, lower_bounds, upper_bounds, mutation_rate):
    child = individual.copy()
    for i in range(len(child)):
        if np.random.random() < mutation_rate:
            sigma = (upper_bounds[i] - lower_bounds[i]) * 0.1
            child[i] += np.random.normal(0, sigma)
            child[i] = np.clip(child[i], lower_bounds[i], upper_bounds[i])
    return child


In [16]:
fitness_history = []


def objective_function(solution):
    params = decode_solution(solution)

    try:
        tf.keras.backend.clear_session()
        gc.collect()
        set_seed(SEED)

        feature_mask = params["feature_mask"]

        X_train_selected = apply_feature_mask(X_train, feature_mask)
        X_val_selected = apply_feature_mask(X_val, feature_mask)

        model = build_lstm_model(
            params=params,
            input_shape=(
                WINDOW_SIZE,
                params["n_selected_features"]
            )
        )

        early_stopping = EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE_OPT,
            restore_best_weights=True
        )

        model.fit(
            X_train_selected,
            y_train,
            validation_data=(X_val_selected, y_val),
            epochs=OPTIMIZATION_EPOCHS,
            batch_size=params["batch_size"],
            callbacks=[early_stopping],
            verbose=0
        )

        y_pred_val = model.predict(
            X_val_selected,
            batch_size=params["batch_size"],
            verbose=0
        )

        y_val_inv = inverse_transform(y_val)
        y_pred_val_inv = inverse_transform(y_pred_val)

        mae = mean_absolute_error(y_val_inv, y_pred_val_inv)

        fitness_history.append({
            "fitness": mae,
            "status": "ok",
            "units_1": params["units_1"],
            "units_2": params["units_2"],
            "dropout_rate": params["dropout_rate"],
            "learning_rate": params["learning_rate"],
            "batch_size": params["batch_size"],
            "optimizer": params["optimizer"],
            "activation": params["activation"],
            "n_layers": params["n_layers"],
            "n_selected_features": params["n_selected_features"],
            "selected_features": ",".join(params["selected_features"])
        })

        del model
        gc.collect()
        tf.keras.backend.clear_session()

        return mae

    except Exception as e:
        fitness_history.append({
            "fitness": 999999,
            "status": "failed",
            "error": str(e)[:300]
        })

        try:
            del model
        except:
            pass

        gc.collect()
        tf.keras.backend.clear_session()

        return 999999

In [17]:
def run_pso(
    objective_function,
    lower_bounds,
    upper_bounds,
    n_particles=20,
    iterations=30,
    w=0.7,
    c1=1.5,
    c2=1.5,
    patience=6,
    min_delta=0.01,
    seed=42
):
    np.random.seed(seed)

    dim = len(lower_bounds)

    positions = np.random.uniform(
        lower_bounds,
        upper_bounds,
        size=(n_particles, dim)
    )

    velocities = np.zeros((n_particles, dim))

    personal_best_positions = positions.copy()
    personal_best_scores = np.array([
        objective_function(pos) for pos in positions
    ])

    best_index = np.argmin(personal_best_scores)
    global_best_position = personal_best_positions[best_index].copy()
    global_best_score = personal_best_scores[best_index]

    convergence = [global_best_score]

    no_improvement_counter = 0

    for iteration in range(iterations):
        print(f"PSO iteration {iteration + 1}/{iterations}")

        previous_best_score = global_best_score

        r1 = np.random.rand(n_particles, dim)
        r2 = np.random.rand(n_particles, dim)

        velocities = (
            w * velocities
            + c1 * r1 * (personal_best_positions - positions)
            + c2 * r2 * (global_best_position - positions)
        )

        positions = positions + velocities
        positions = np.clip(positions, lower_bounds, upper_bounds)

        for i in range(n_particles):
            score = objective_function(positions[i])

            if score < personal_best_scores[i]:
                personal_best_scores[i] = score
                personal_best_positions[i] = positions[i].copy()

                if score < global_best_score:
                    global_best_score = score
                    global_best_position = positions[i].copy()

        improvement = previous_best_score - global_best_score

        if improvement > min_delta:
            no_improvement_counter = 0
        else:
            no_improvement_counter += 1

        convergence.append(global_best_score)

        print("Best MAE:", global_best_score)
        print("Improvement:", improvement)
        print("No improvement counter:", no_improvement_counter)

        if no_improvement_counter >= patience:
            print("Early stopping PSO: no relevant improvement.")
            break

    return global_best_position, global_best_score, convergence

In [18]:
pso_best_solution, pso_best_score, pso_convergence = run_pso(
    objective_function=objective_function,
    lower_bounds=LOWER_BOUNDS,
    upper_bounds=UPPER_BOUNDS,
    n_particles=6,
    iterations=10,
    patience=3,
    min_delta=0.01,
    seed=SEED
)

PSO iteration 1/10
Best MAE: 66.15661494047457
Improvement: 0.0
No improvement counter: 1
PSO iteration 2/10
Best MAE: 59.04862461410731
Improvement: 7.107990326367258
No improvement counter: 0
PSO iteration 3/10
Best MAE: 59.04862461410731
Improvement: 0.0
No improvement counter: 1
PSO iteration 4/10
Best MAE: 59.04862461410731
Improvement: 0.0
No improvement counter: 2
PSO iteration 5/10
Best MAE: 59.04862461410731
Improvement: 0.0
No improvement counter: 3
Early stopping PSO: no relevant improvement.


In [ ]:
def run_ga(
    objective_function,
    lower_bounds,
    upper_bounds,
    population_size=20,
    generations=30,
    mutation_rate=0.2,
    elite_size=2,
    patience=6,
    min_delta=0.01,
    seed=42
):
    def tournament_selection(population, fitness, k=3):
        indices = np.random.choice(len(population), size=k, replace=False)
        best_idx = indices[np.argmin(fitness[indices])]
        return population[best_idx].copy()

    def crossover(parent1, parent2, alpha=0.5):
        d = np.abs(parent1 - parent2)
        lo = np.minimum(parent1, parent2) - alpha * d
        hi = np.maximum(parent1, parent2) + alpha * d
        child1 = lo + np.random.random(len(parent1)) * (hi - lo)
        child2 = lo + np.random.random(len(parent2)) * (hi - lo)
        return child1, child2

    def mutate(individual, lower_bounds, upper_bounds, mutation_rate):
        child = individual.copy()
        for i in range(len(child)):
            if np.random.random() < mutation_rate:
                sigma = (upper_bounds[i] - lower_bounds[i]) * 0.1
                child[i] += np.random.normal(0, sigma)
                child[i] = np.clip(child[i], lower_bounds[i], upper_bounds[i])
        return child

    np.random.seed(seed)
    dim = len(lower_bounds)
    population = np.random.uniform(lower_bounds, upper_bounds, size=(population_size, dim))
    fitness = np.array([objective_function(individual) for individual in population])
    best_index = np.argmin(fitness)
    best_solution = population[best_index].copy()
    best_score = fitness[best_index]
    convergence = [best_score]
    no_improvement_counter = 0

    for generation in range(generations):
        print(f"GA generation {generation + 1}/{generations}")
        previous_best_score = best_score
        new_population = []
        elite_indices = np.argsort(fitness)[:elite_size]
        for idx in elite_indices:
            new_population.append(population[idx].copy())
        while len(new_population) < population_size:
            parent1 = tournament_selection(population, fitness)
            parent2 = tournament_selection(population, fitness)
            child1, child2 = crossover(parent1, parent2)
            child1 = mutate(child1, lower_bounds, upper_bounds, mutation_rate)
            child2 = mutate(child2, lower_bounds, upper_bounds, mutation_rate)
            new_population.append(child1)
            if len(new_population) < population_size:
                new_population.append(child2)
        population = np.array(new_population)
        fitness = np.array([objective_function(individual) for individual in population])
        current_best_index = np.argmin(fitness)
        if fitness[current_best_index] < best_score:
            best_score = fitness[current_best_index]
            best_solution = population[current_best_index].copy()
        improvement = previous_best_score - best_score
        if improvement > min_delta:
            no_improvement_counter = 0
        else:
            no_improvement_counter += 1
        convergence.append(best_score)
        print(f"Best MAE: {best_score}  Improvement: {improvement}  No-impr: {no_improvement_counter}")
        if no_improvement_counter >= patience:
            print("Early stopping GA.")
            break

    return best_solution, best_score, convergence

print("run_ga definida (helpers incluidos)")


In [ ]:
def tournament_selection(population, fitness, k=3):
    indices = np.random.choice(len(population), size=k, replace=False)
    best_idx = indices[np.argmin(fitness[indices])]
    return population[best_idx].copy()

def crossover(parent1, parent2, alpha=0.5):
    d = np.abs(parent1 - parent2)
    lo = np.minimum(parent1, parent2) - alpha * d
    hi = np.maximum(parent1, parent2) + alpha * d
    child1 = lo + np.random.random(len(parent1)) * (hi - lo)
    child2 = lo + np.random.random(len(parent2)) * (hi - lo)
    return child1, child2

def mutate(individual, lower_bounds, upper_bounds, mutation_rate):
    child = individual.copy()
    for i in range(len(child)):
        if np.random.random() < mutation_rate:
            sigma = (upper_bounds[i] - lower_bounds[i]) * 0.1
            child[i] += np.random.normal(0, sigma)
            child[i] = np.clip(child[i], lower_bounds[i], upper_bounds[i])
    return child

ga_best_solution, ga_best_score, ga_convergence = run_ga(
    objective_function=objective_function,
    lower_bounds=LOWER_BOUNDS,
    upper_bounds=UPPER_BOUNDS,
    population_size=20,
    generations=30,
    mutation_rate=0.2,
    elite_size=2,
    patience=6,
    min_delta=0.01,
    seed=SEED
)

In [ ]:
def params_to_result_row(optimizer_name, params, score, total_time):
    return {
        "Optimizer": optimizer_name,
        "Validation_MAE": score,
        "Optimization_Time_Seconds": total_time,
        "units_1": params["units_1"],
        "units_2": params["units_2"],
        "dropout_rate": params["dropout_rate"],
        "learning_rate": params["learning_rate"],
        "batch_size": params["batch_size"],
        "optimizer": params["optimizer"],
        "activation": params["activation"],
        "n_layers": params["n_layers"],
        "n_selected_features": params["n_selected_features"],
        "selected_features": ",".join(params["selected_features"])
    }


optimization_results = [
    params_to_result_row(
        "PSO",
        pso_params,
        pso_best_score,
        pso_time
    ),
    params_to_result_row(
        "GA",
        ga_params,
        ga_best_score,
        ga_time
    )
]

optimization_results_df = pd.DataFrame(optimization_results)

optimization_results_df

In [ ]:
optimization_results_path = os.path.join(
    RESULTS_DIR,
    "feature_selection_optimization_results.csv"
)

optimization_results_df.to_csv(
    optimization_results_path,
    index=False
)

print("Saved:", optimization_results_path)

In [ ]:
fitness_history_df = pd.DataFrame(fitness_history)

fitness_history_path = os.path.join(
    RESULTS_DIR,
    "feature_selection_fitness_history.csv"
)

fitness_history_df.to_csv(
    fitness_history_path,
    index=False
)

print("Saved:", fitness_history_path)

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(pso_convergence, label="PSO")
plt.plot(ga_convergence, label="GA")

plt.title("Feature Selection Optimization Convergence")
plt.xlabel("Iteration")
plt.ylabel("Best Validation MAE")
plt.legend()
plt.grid(True)

convergence_path = os.path.join(
    FIGURES_DIR,
    "feature_selection_convergence.png"
)

plt.savefig(
    convergence_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:", convergence_path)

In [ ]:
def train_final_model(optimizer_name, params):
    tf.keras.backend.clear_session()
    gc.collect()
    set_seed(SEED)

    feature_mask = params["feature_mask"]

    X_train_selected = apply_feature_mask(X_train, feature_mask)
    X_val_selected = apply_feature_mask(X_val, feature_mask)
    X_test_selected = apply_feature_mask(X_test, feature_mask)

    model = build_lstm_model(
        params=params,
        input_shape=(
            WINDOW_SIZE,
            params["n_selected_features"]
        )
    )

    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE_FINAL,
        restore_best_weights=True
    )

    start_time = time.time()

    history = model.fit(
        X_train_selected,
        y_train,
        validation_data=(X_val_selected, y_val),
        epochs=FINAL_EPOCHS,
        batch_size=params["batch_size"],
        callbacks=[early_stopping],
        verbose=1
    )

    training_time = time.time() - start_time

    y_pred_test = model.predict(
        X_test_selected,
        batch_size=params["batch_size"],
        verbose=0
    )

    y_test_inv = inverse_transform(y_test)
    y_pred_test_inv = inverse_transform(y_pred_test)

    overall_metrics = calculate_metrics(
        y_test_inv,
        y_pred_test_inv
    )

    daylight_metrics = calculate_daylight_metrics(
        y_test_inv,
        y_pred_test_inv,
        daylight_test
    )

    result = {
        "Method": f"{optimizer_name}_FeatureSelection",
        "Optimizer": optimizer_name,
        "Test_MAE": overall_metrics["MAE"],
        "Test_RMSE": overall_metrics["RMSE"],
        "Test_R2": overall_metrics["R2"],
        "Daylight_MAE": daylight_metrics["MAE"],
        "Daylight_RMSE": daylight_metrics["RMSE"],
        "Daylight_R2": daylight_metrics["R2"],
        "Training_Time_Seconds": training_time,
        "Epochs": len(history.history["loss"]),
        "units_1": params["units_1"],
        "units_2": params["units_2"],
        "dropout_rate": params["dropout_rate"],
        "learning_rate": params["learning_rate"],
        "batch_size": params["batch_size"],
        "optimizer": params["optimizer"],
        "activation": params["activation"],
        "n_layers": params["n_layers"],
        "n_selected_features": params["n_selected_features"],
        "selected_features": ",".join(params["selected_features"])
    }

    model_path = os.path.join(
        MODELS_DIR,
        f"{optimizer_name}_feature_selection.keras"
    )

    model.save(model_path)

    return result, history, y_test_inv, y_pred_test_inv

In [ ]:
final_results = []
final_histories = {}
final_predictions = {}

for optimizer_name, params in {
    "PSO": pso_params,
    "GA": ga_params
}.items():

    result, history, y_true, y_pred = train_final_model(
        optimizer_name,
        params
    )

    final_results.append(result)

    final_histories[optimizer_name] = history

    final_predictions[optimizer_name] = {
        "y_true": y_true,
        "y_pred": y_pred
    }

In [ ]:
final_results_df = pd.DataFrame(final_results)

final_results_df

In [ ]:
def plot_predictions(y_true, y_pred, title, n_points=300):
    plt.figure(figsize=(15, 5))

    plt.plot(y_true[:n_points], label="Real")
    plt.plot(y_pred[:n_points], label="Predicted")

    plt.title(title)
    plt.xlabel("Time step")
    plt.ylabel("Solar Radiation")
    plt.legend()
    plt.grid(True)

    plt.show()

In [ ]:
for optimizer_name, data in final_predictions.items():
    plot_predictions(
        data["y_true"],
        data["y_pred"],
        f"{optimizer_name} Feature Selection Prediction"
    )

In [ ]:
selected_features_summary = final_results_df[
    [
        "Optimizer",
        "n_selected_features",
        "selected_features",
        "Daylight_MAE",
        "Daylight_RMSE",
        "Daylight_R2"
    ]
]

selected_features_summary

In [ ]:
selected_features_path = os.path.join(
    RESULTS_DIR,
    "selected_features_summary.csv"
)

selected_features_summary.to_csv(
    selected_features_path,
    index=False
)

print("Saved:", selected_features_path)